# ਲੈਸਨ 05 - ਏਜੇਂਟਿਕ RAG


## ਸੈਟਅਪ

ਇਹ ਨੋਟਬੁੱਕ Microsoft Agent Framework ਦੀ ਵਰਤੋਂ ਕਰਦਿਆਂ Agentic RAG (Retrieval-Augmented Generation) ਪੈਟਰਨ ਨੂੰ ਦਰਸਾਉਂਦਾ ਹੈ।

**ਲਾਜ਼ਮੀ ਚੀਜ਼ਾਂ:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — ਤੁਹਾਡਾ Azure AI Search ਸੇਵਾ ਐਂਡਪੌਇੰਟ
- `AZURE_SEARCH_API_KEY` — ਤੁਹਾਡੀ Azure AI Search API ਕੀ
- ਐਨਵਾਇਰਨਮੈਂਟ ਵੈਰੀਏਬਲਾਂ ਰਾਹੀਂ ਸੰਰਚਿਤ Azure OpenAI ਡਿਪਲੌਇਮੈਂਟ
- Azure CLI ਪ੍ਰਮਾਣਿਕ ਕੀਤਾ ਗਿਆ (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Agentic RAG ਕੀ ਹੈ?

ਪੰਰਪਰਾਗਤ RAG ਇੱਕ ਨਿਸ਼ਚਿਤ ਪਾਈਪਲਾਈਨ ਦੀ ਪਾਲਣਾ ਕਰਦਾ ਹੈ: ਦਸਤਾਵੇਜ਼ ਪ੍ਰਾਪਤ ਕਰੋ, ਫਿਰ ਜਵਾਬ ਬਣਾਓ। **Agentic RAG** ਇਸ ਤੋਂ ਅੱਗੇ ਵਧਦਾ ਹੈ ਜਿੱਥੇ ਏਜੰਟ ਨੂੰ ਸੁਤੰਤਰਤਾ ਦਿੱਤੀ ਜਾਂਦੀ ਹੈ ਕਿ ਉਹ ਫੈਸਲਾ ਕਰ ਸਕੇ **ਕਦੋਂ** ਅਤੇ **ਕਿਵੇਂ** ਜਾਣਕਾਰੀ ਪ੍ਰਾਪਤ ਕਰਨੀ ਹੈ।

Agentic RAG ਨਾਲ, ਏਜੰਟ ਕਰ ਸਕਦਾ ਹੈ:
- **ਫੈਸਲਾ ਕਰੋ** ਕਿ ਜਵਾਬ ਦੇਣ ਤੋਂ ਪਹਿਲਾਂ ਪ੍ਰਾਪਤੀ ਦੀ ਲੋੜ ਹੈ ਜਾਂ ਨਹੀਂ
- **ਚੁਣੋ** ਕਿ ਕਿਹੜਾ ਡਾਟਾ ਸਰੋਤ ਜਾਂ ਸੰਦ ਪੁੱਛਨਾ ਹੈ
- **ਮੁਲਾਂਕਣ ਕਰੋ** ਪ੍ਰਾਪਤ ਨਤੀਜੇ ਅਤੇ ਜੇ ਪਹਿਲੀ ਕੋਸ਼ਿਸ਼ ਅਪੂਰਣ ਹੈ ਤਾਂ ਫਾਲੋ-ਅਪ ਪ੍ਰਾਪਤੀ ਕਰੋ
- **ਝੁੜੋ** ਵੱਖ-ਵੱਖ ਪ੍ਰਾਪਤੀ ਕਦਮਾਂ ਤੋਂ ਮਿਲੀ ਜਾਣਕਾਰੀ ਨੂੰ ਇਕ ਸੰਗਠਿਤ ਜਵਾਬ ਵਿੱਚ

ਇਸ ਨਾਲ ਏਜੰਟ ਇੱਕ ਸਥਿਰ retrieve-then-generate ਪਾਈਪਲਾਈਨ ਨਾਲੋਂ ਵੱਧ ਲਚਕੀਲਾ ਅਤੇ ਸਹੀ ਬਣ ਜਾਂਦਾ ਹੈ।


## ਇੱਕ ਖੋਜ ਸੰਦ ਬਣਾਉਂਦਾ

Agentic RAG ਵਿੱਚ, ਬਾਹਰੀ ਡਾਟਾ ਸਰੋਤਾਂ ਨੂੰ **ਸੰਦਾਂ** ਵਜੋਂ ਲਪੇਟਿਆ ਜਾਂਦਾ ਹੈ ਜਿਸਨੂੰ ਏਜੰਟ ਜਰੂਰਤ ਪੈਣ 'ਤੇ ਸੱਦਾ ਦੇ ਸਕਦਾ ਹੈ। ਇਸ ਨਾਲ ਏਜੰਟ ਰੀਟਰੀਵਲ ਨੂੰ ਸਿਰਫ ਇੱਕ ਹੋਰ ਕਾਰਵਾਈ ਵਜੋਂ ਲੈ ਸਕਦਾ ਹੈ, ਬਜਾਏ ਕਿ ਕੋਈ ਜ਼ਰੂਰੀ ਕਦਮ ਹੋਣ ਦੇ।

ਥੱਲੇ ਅਸੀਂ ਇੱਕ ਯਾਤਰਾ ਗਿਆਨ ਆਧਾਰ ਪਰਿਭਾਸ਼ਿਤ ਕਰਦੇ ਹਾਂ ਅਤੇ ਇਸਨੂੰ ਇੱਕ ਸੰਦ ਵਜੋਂ ਪ੍ਰਗਟ ਕਰਦੇ ਹਾਂ ਜਿਸਨੂੰ ਏਜੰਟ ਮੰਜ਼ਿਲ ਜਾਣਕਾਰੀ ਲਈ ਕਾਲ ਕਰ ਸਕਦਾ ਹੈ।


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAG ਏਜੰਟ ਬਣਾਉਣਾ

ਹੁਣ ਅਸੀਂ ਇੱਕ ਐਜੰਟ ਬਣਾਉਂਦੇ ਹਾਂ ਜਿਸਨੂੰ **ਜਵਾਬ ਦੇਣ ਤੋਂ ਪਹਿਲਾਂ ਸਦਾ ਜਾਣਕਾਰੀ ਪ੍ਰਾਪਤ ਕਰਨ ਲਈ ਹੁਕਮ ਦਿੱਤਾ ਗਿਆ ਹੈ**। ਐਜੰਟ ਆਪਣੇ ਜਵਾਬਾਂ ਨੂੰ ਸਿੱਖਿਆ ਡੇਟਾ 'ਤੇ ਭਰੋਸਾ ਕਰਨ ਦੀ ਬਜਾਇ ਗਿਆਨ ਅਧਾਰ ਵਿੱਚ ਸਥਾਪਿਤ ਕਰਨ ਲਈ `search_travel_knowledge` ਟੂਲ ਦੀ ਵਰਤੋਂ ਕਰਦਾ ਹੈ।


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Iterative Retrieval — The Maker-Checker Pattern

ਏਜੰਟਿਕ RAG ਦਾ ਇੱਕ ਮੁੱਖ ਫਾਇਦਾ ਹੈ **ਆਵਰਤੀ ਪ੍ਰਾਪਤੀ**। ਏਜੰਟ ਆਪਣੇ ਪਹਿਲੇ ਨਤੀਜਿਆਂ ਦੀ ਜਾਂਚ, ਸੁਧਾਰ, ਜਾਂ ਵਧੇਰੇ ਜਾਣਕਾਰੀ ਹਾਸਲ ਕਰਨ ਲਈ ਕਈ ਵਾਰੀ ਖੋਜ ਕਰ ਸਕਦਾ ਹੈ — ਜਿਵੇਂ ਕਿ "ਮੇਕਰ-ਚੈੱਕਰ" ਵਰਕਫਲੋ:

1. **ਮੇਕਰ ਕਦਮ**: ਏਜੰਟ ਸਭ ਤੋਂ ਪਹਿਲਾਂ ਜਾਣਕਾਰੀ ਪ੍ਰਾਪਤ ਕਰਦਾ ਹੈ ਅਤੇ ਜਵਾਬ ਦਾ ਮਸੌਦਾ ਬਣਾਉਂਦਾ ਹੈ।
2. **ਚੈੱਕਰ ਕਦਮ**: ਏਜੰਟ ਵਿਸਥਾਰਕ ਜਾਂਚ ਲਈ ਵਾਧੂ ਪ੍ਰਾਪਤੀਆਂ ਕਰਦਾ ਹੈ ਜਾਂ ਖ਼ਾਮੀਆਂ ਭਰਨ ਲਈ।

ਹੇਠਾਂ, ਏਜੰਟ ਨੂੰ ਇੱਕ ਐਸਾ ਸਵਾਲ ਪੁੱਛਿਆ ਗਿਆ ਹੈ ਜਿਸ ਲਈ ਕਈ ਮੰਜ਼ਿਲਾਂ ਦੀ ਤੁਲਨਾ ਕਰਨ ਦੀ ਲੋੜ ਹੈ, ਜਿਸ ਨਾਲ ਉਹ ਕਈ ਵਾਰੀ ਖੋਜ ਕਰਨ ਲਈ ਪ੍ਰੇਰਿਤ ਹੁੰਦਾ ਹੈ।


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Summary

ਇਸ ਪਾਠ ਵਿੱਚ ਤੁਸੀਂ Microsoft Agent Framework ਦੀ ਵਰਤੋਂ ਕਰਕੇ **Agentic RAG** ਸਿਸਟਮ ਬਣਾਉਣਾ ਸਿੱਖਿਆ:

- **Agentic RAG** ਏਜੰਟਾਂ ਨੂੰ ਸਵੈਚਾਲਿਤ ਤਰੀਕੇ ਨਾਲ ਜਾਣਕਾਰੀ ਪ੍ਰਾਪਤ ਕਰਨ ਦਾ ਫੈਸਲਾ ਕਰਨ ਦਿੰਦਾ ਹੈ, ਜਿਸ ਨਾਲ ਪ੍ਰਾਪਤੀ ਦ੍ਰਿੜ੍ਹ ਨਹੀਂ ਸਗੋਂ ਗਤੀਸ਼ੀਲ ਬਣ ਜਾਂਦੀ ਹੈ।
- **ਟੂਲਾਂ ਨੂੰ ਡਾਟਾ ਸਰੋਤ ਵਜੋਂ ਵਰਤਣਾ**: ਬਾਹਰੀ ਗਿਆਨ ਆਧਾਰ (ਜਿਵੇਂ Azure AI Search) ਨੂੰ ਟੂਲਾਂ ਵਜੋਂ ਪੇਸ਼ ਕੀਤਾ ਜਾਂਦਾ ਹੈ, ਜਿਨ੍ਹਾਂ ਨੂੰ ਏਜੰਟ ਕਾਲ ਕਰ ਸਕਦਾ ਹੈ।
- **ਦੁਹਰਾਈ ਵਾਲੀ ਪ੍ਰਾਪਤੀ**: maker-checker ਪੈਟਰਨ ਏਜੰਟ ਨੂੰ ਕਈ ਵਾਰੀ ਜਾਣਕਾਰੀ ਲੈਣ, ਜਾਂਚਣ ਅਤੇ ਸੁਧਾਰ ਕਰਨ ਦੀ ਆਗਿਆ ਦਿੰਦਾ ਹੈ, ਫਿਰ ਅੰਤਿਮ ਜਵਾਬ ਤਿਆਰ ਹੁੰਦਾ ਹੈ।

ਉਤਪਾਦਨ ਵਿੱਚ, ਤੁਸੀਂ ਯਾਦ ਵਿੱਚ ਮੌਜੂਦ `TRAVEL_KNOWLEDGE_BASE` ਨੂੰ ਸੱਚੇ Azure AI Search ਇੰਡੈਕਸ ਨਾਲ ਬਦਲੋਗੇ ਤਾਂ ਜੋ ਵੱਡੇ ਪੱਧਰ ਤੇ ਯਾਤਰਾ ਦਸਤਾਵੇਜ਼ਾਂ ਦੀ ਪ੍ਰਾਪਤੀ ਕੀਤੀ ਜਾ ਸਕੇ।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ਇਨਕਾਰ**:  
ਇਹ ਦਸਤਾਵੇਜ਼ ਏਆਈ ਅਨੁਵਾਦ ਸੇਵਾ [Co-op Translator](https://github.com/Azure/co-op-translator) ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਅਨੁਵਾਦ ਕੀਤਾ ਗਿਆ ਹੈ। ਜਦੋਂ ਕਿ ਅਸੀਂ ਸਹੀਤਾ ਲਈ ਕੋਸ਼ਿਸ਼ ਕਰਦੇ ਹਾਂ, ਕਿਰਪਾ ਕਰਕੇ ਧਿਆਨ ਵਿੱਚ ਰੱਖੋ ਕਿ ਆਟੋਮੈਟਿਕ ਅਨੁਵਾਦਾਂ ਵਿੱਚ ਗਲਤੀਆਂ ਜਾਂ ਅਸਮੱਥਤਾਵਾਂ ਹੋ ਸਕਦੀਆਂ ਹਨ। ਮੂਲ ਦਸਤਾਵੇਜ਼ ਆਪਣੀ ਮੂਲ ਭਾਸ਼ਾ ਵਿੱਚ ਪ੍ਰਮਾਣਿਕ ਸਰੋਤ ਮੰਨਿਆ ਜਾਣਾ ਚਾਹੀਦਾ ਹੈ। ਮਹੱਤਵਪੂਰਨ ਜਾਣਕਾਰੀ ਲਈ, ਪੇਸ਼ੇਵਰ ਮਨੁੱਖੀ ਅਨੁਵਾਦ ਦੀ ਸਿਫਾਰਿਸ਼ ਕੀਤੀ ਜਾਂਦੀ ਹੈ। ਅਸੀਂ ਇਸ ਅਨੁਵਾਦ ਦੀ ਵਰਤੋਂ ਨਾਲ ਪੈਦਾ ਹੋਣ ਵਾਲੀਆਂ ਕਿਸੇ ਵੀ ਗਲਤਫਹਿਮੀਆਂ ਜਾਂ ਗਲਤ ਸਮਝਣਾ ਲਈ ਜ਼ਿੰਮੇਵਾਰ ਨਹੀਂ ਹਾਂ।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
